In [1]:
from pyspark.sql import SparkSession

KAFKA_PKG = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.2"
SEDONA_PKG = "org.apache.sedona:sedona-spark-shaded-4.0_2.13:1.8.1"
GEOTOOLS_PKG = "org.datasyslab:geotools-wrapper:1.8.1-33.1"
ICEBERG_PKG = "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1"

HADOOP_AWS = "org.apache.hadoop:hadoop-aws:3.4.1"
AWS_SDK_PKG = "software.amazon.awssdk:bundle:2.29.38"

spark = SparkSession.builder \
    .appName("NYC_Environmental_Impact") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", f"{KAFKA_PKG},{SEDONA_PKG},{GEOTOOLS_PKG},{ICEBERG_PKG},{HADOOP_AWS},{AWS_SDK_PKG}") \
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "org.apache.sedona.sql.SedonaSqlExtensions") \
    .config("spark.sql.catalog.local",           "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type",      "rest") \
    .config("spark.sql.catalog.local.uri",       "http://rest-catalog:8181") \
    .config("spark.sql.catalog.local.warehouse", "s3a://business-data/") \
    .config("spark.sql.catalog.local.s3.access-key-id",     "admin") \
    .config("spark.sql.catalog.local.s3.secret-access-key", "password") \
    .config("spark.sql.catalog.local.s3.region",            "us-east-1") \
    .config("spark.sql.catalog.local.s3.endpoint",          "http://minio:9000") \
    .config("spark.sql.catalog.local.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.endpoint",          "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",        "admin") \
    .config("spark.hadoop.fs.s3a.secret.key",        "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.driver.host",        "jupyter-pyspark") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.driver.memory",      "4g") \
    .config("spark.executor.memory",    "4g") \
    .config("spark.sql.streaming.statefulOperator.allowMultiple", "true") \
    .getOrCreate()

print(f"Spark Session Ready! Version: {spark.version}")

Spark Session Ready! Version: 4.0.2


In [2]:
for q in spark.streams.active:
    q.stop()

# Full Reset Checkpoint
spark.sql("DROP TABLE IF EXISTS local.db.enriched_traffic")

DataFrame[]

In [3]:
from pyspark.sql.functions import (
    col, expr, from_json, lit, to_timestamp, udf
)
from pyspark.sql.types import (
    ArrayType, BooleanType, DoubleType, IntegerType,
    LongType, StringType, StructField, StructType, TimestampType
)
import h3
from pyspark.sql.functions import coalesce, regexp_extract
from pyspark.sql.functions import col, expr, from_json, lit, to_timestamp, to_utc_timestamp

KAFKA_BOOTSTRAP = "redpanda:29092"

# 15 minutes balances tolerance for delayed traffic/AQ events vs. state growth.
# Tune this single value if your source lateness profile changes.
WATERMARK_DELAY = "15 minutes"
# Keep weather window aligned with traffic/AQ during local debugging.
WEATHER_WATERMARK_DELAY = "15 minutes"
JOIN_WINDOW_MINUTES = 15
WEATHER_JOIN_WINDOW_MINUTES = 15
STREAM_TRIGGER_INTERVAL = "30 seconds"

# Bump this suffix when stateful-stream schemas change.
CHECKPOINT_PATH = "s3a://business-data/checkpoints/local.db.enriched_traffic_v4"
BUSINESS_CHECKPOINT_PATH = CHECKPOINT_PATH

In [4]:
traffic_schema = StructType([
    StructField("id",               StringType(), True),
    StructField("status",           StringType(), True),
    StructField("speed",            StringType(), True),
    StructField("travel_time",      StringType(), True),
    StructField("link_name",        StringType(), True),
    StructField("borough",          StringType(), True),
    StructField("from_street",      StringType(), True),
    StructField("to_street",        StringType(), True),
    StructField("data_as_of",       StringType(), True),
    StructField("link_points",      StringType(), True),
    StructField("encoded_poly_line", StringType(), True),
])

openaq_schema = StructType([
    StructField("sensor_id",     StringType(), True),
    StructField("source",        StringType(), True),
    StructField("lat",           DoubleType(), True),
    StructField("lon",           DoubleType(), True),
    StructField("pm25",          DoubleType(), True),
    StructField("timestamp",     StringType(), True),
    StructField("location_id",   LongType(),   True),
    StructField("location_name", StringType(), True),
    StructField("latitude",      DoubleType(), True),
    StructField("longitude",     DoubleType(), True),
    StructField("parameter",     StringType(), True),
    StructField("value",         DoubleType(), True),
    StructField("unit",          StringType(), True),
    StructField("datetime_utc",  StringType(), True),
])

purpleair_schema = StructType([
    StructField("sensor_id",       StringType(), True),
    StructField("source",          StringType(), True),
    StructField("lat",             DoubleType(), True),
    StructField("lon",             DoubleType(), True),
    StructField("pm25",            DoubleType(), True),
    StructField("timestamp",       StringType(), True),
    StructField("sensor_index",    LongType(),   True),
    StructField("name",            StringType(), True),
    StructField("latitude",        DoubleType(), True),
    StructField("longitude",       DoubleType(), True),
    StructField("pm2.5_10minute",  DoubleType(), True),
    StructField("temperature",     DoubleType(), True),
    StructField("humidity",        DoubleType(), True),
])

weather_schema = StructType([
    StructField("number",          StringType(),  True),
    StructField("name",            StringType(),  True),
    StructField("startTime",       StringType(),  True),
    StructField("endTime",         StringType(),  True),
    StructField("isDaytime",       BooleanType(), True),
    StructField("temperature",     StringType(),  True),
    StructField("temperatureUnit", StringType(),  True),
    StructField("temperatureTrend", StringType(),  True),
    StructField("windSpeed",       StringType(),  True),
    StructField("windDirection",   StringType(),  True),
    StructField("shortForecast",   StringType(),  True),
    StructField("detailedForecast", StringType(),  True),
    StructField("probabilityOfPrecipitation", StructType([
        StructField("unitCode", StringType(),  True),
        StructField("value",    StringType(),  True),
    ]), True),
])

In [5]:
def read_kafka_json(topic_name: str, json_schema: StructType):
    parser_schema = StructType(json_schema.fields + [StructField("_corrupt_json", StringType(), True)])

    parsed_df = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
        .option("subscribe", topic_name)
        .option("startingOffsets", "earliest")
        .option("maxOffsetsPerTrigger", 1000)
        .load()
        .select(
            col("value").cast("string").alias("raw_payload"),
            col("timestamp").alias("kafka_timestamp"),
            lit(topic_name).alias("source_topic"),
            from_json(
                col("value").cast("string"),
                parser_schema,
                {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_json"},
            ).alias("payload"),
        )
    )

    valid_df = (
        parsed_df
        .where(col("payload._corrupt_json").isNull())
        .select("payload.*", "kafka_timestamp")
        .drop("_corrupt_json")
    )

    malformed_df = (
        parsed_df
        .where(col("payload._corrupt_json").isNotNull())
        .select("source_topic", "kafka_timestamp", "raw_payload")
    )

    return valid_df, malformed_df


@udf(returnType=StringType())
def latlon_to_h3(lat, lon):
    if lat is None or lon is None:
        return None
    try:
        # return h3.latlng_to_cell(float(lat), float(lon), 8)
        return h3.latlng_to_cell(float(lat), float(lon), 7)
    except AttributeError:
        # return h3.geo_to_h3(float(lat), float(lon), 8)
        return h3.geo_to_h3(float(lat), float(lon), 7)
    except Exception:
        return None

In [6]:
STREAM_TOPICS = {
    "traffic": "nyc_traffic_raw",
    "openaq": "nyc_openaq_raw",
    "purpleair": "nyc_purpleair_raw",
    "weather": "nyc_weather_raw",
}

for stream_name, topic in STREAM_TOPICS.items():
    print(f"[stream_config] stream={stream_name} topic={topic} bootstrap={KAFKA_BOOTSTRAP}")
print(f"[stream_config] watermark_delay={WATERMARK_DELAY}")
print(f"[stream_config] weather_watermark_delay={WEATHER_WATERMARK_DELAY}")
print(f"[join_config] traffic_aq_window_minutes={JOIN_WINDOW_MINUTES}")
print(f"[join_config] weather_window_minutes={WEATHER_JOIN_WINDOW_MINUTES}")
print(f"[stream_config] trigger_interval={STREAM_TRIGGER_INTERVAL}")
print(f"[stream_config] business_checkpoint={BUSINESS_CHECKPOINT_PATH}")

traffic_raw_df, traffic_malformed_df = read_kafka_json(STREAM_TOPICS["traffic"], traffic_schema)
openaq_raw_df, openaq_malformed_df = read_kafka_json(STREAM_TOPICS["openaq"], openaq_schema)
purpleair_raw_df, purpleair_malformed_df = read_kafka_json(STREAM_TOPICS["purpleair"], purpleair_schema)
weather_raw_df, weather_malformed_df = read_kafka_json(STREAM_TOPICS["weather"], weather_schema)

malformed_json_df = (
    traffic_malformed_df
    .unionByName(openaq_malformed_df)
    .unionByName(purpleair_malformed_df)
    .unionByName(weather_malformed_df)
)

NYC_LAT_MIN, NYC_LAT_MAX = 40.4774, 40.9176
NYC_LON_MIN, NYC_LON_MAX = -74.2591, -73.7004
TRAFFIC_SPEED_MAX_MPH = 100.0
AQ_PM25_MAX_UGM3 = 2000.0

traffic_df = (
    traffic_raw_df
    .withColumn("speed", col("speed").cast(DoubleType()))
    .withColumn("travel_time", col("travel_time").cast(DoubleType()))
    .withColumn("first_link_point", expr("trim(element_at(split(link_points, ' '), 1))"))
    .withColumn("traffic_lat", expr("cast(element_at(split(first_link_point, ','), 1) as double)"))
    .withColumn("traffic_lon", expr("cast(element_at(split(first_link_point, ','), 2) as double)"))
    .withColumn("traffic_geom", expr("ST_Point(traffic_lon, traffic_lat)"))
    .withColumn(
        "traffic_event_ts",
        to_utc_timestamp(
            to_timestamp(col("data_as_of"), "yyyy-MM-dd'T'HH:mm:ss.SSS"),
            "America/New_York",
        ),
    )
    .filter(col("id").isNotNull())
    .filter(col("traffic_event_ts").isNotNull())
    .filter(col("traffic_lat").isNotNull() & col("traffic_lon").isNotNull())
    .filter(
        (col("traffic_lat") >= NYC_LAT_MIN)
        & (col("traffic_lat") <= NYC_LAT_MAX)
        & (col("traffic_lon") >= NYC_LON_MIN)
        & (col("traffic_lon") <= NYC_LON_MAX)
    )
    .filter(col("speed").isNotNull() & col("speed").between(0.0, TRAFFIC_SPEED_MAX_MPH))
    .withColumn("h3_index", latlon_to_h3(col("traffic_lat"), col("traffic_lon")))
    .filter(col("h3_index").isNotNull())
    .withWatermark("traffic_event_ts", WATERMARK_DELAY)
)

openaq_df = (
    openaq_raw_df
    .withColumn("aq_event_ts", coalesce(to_timestamp(col("timestamp")), to_timestamp(col("datetime_utc"))).cast(TimestampType()))
    .withColumn("aq_lat_norm", coalesce(col("lat").cast(DoubleType()), col("latitude").cast(DoubleType())))
    .withColumn("aq_lon_norm", coalesce(col("lon").cast(DoubleType()), col("longitude").cast(DoubleType())))
    .withColumn("aq_pm25_norm", coalesce(col("pm25").cast(DoubleType()), col("value").cast(DoubleType())))
    .withColumn("h3_index", latlon_to_h3(col("aq_lat_norm"), col("aq_lon_norm")))
    .select(
        coalesce(col("source"), lit("openaq")).alias("aq_source"),
        coalesce(
            col("sensor_id"),
            col("location_id").cast(StringType()),
            col("location_name"),
        ).alias("aq_location_id"),
        col("location_name").alias("aq_location_name"),
        col("aq_event_ts"),
        col("kafka_timestamp"),
        col("aq_lat_norm").alias("aq_lat"),
        col("aq_lon_norm").alias("aq_lon"),
        col("h3_index"),
        col("aq_pm25_norm").alias("aq_pm25_ugm3"),
        lit(None).cast(DoubleType()).alias("aq_temperature"),
        lit(None).cast(DoubleType()).alias("aq_humidity"),
    )
    .filter(col("aq_event_ts").isNotNull())
)

purpleair_df = (
    purpleair_raw_df
    .withColumn("aq_event_ts", coalesce(to_timestamp(col("timestamp")), col("kafka_timestamp").cast(TimestampType())))
    .withColumn("aq_lat_norm", coalesce(col("lat").cast(DoubleType()), col("latitude").cast(DoubleType())))
    .withColumn("aq_lon_norm", coalesce(col("lon").cast(DoubleType()), col("longitude").cast(DoubleType())))
    .withColumn("aq_pm25_norm", coalesce(col("pm25").cast(DoubleType()), col("`pm2.5_10minute`").cast(DoubleType())))
    .withColumn("h3_index", latlon_to_h3(col("aq_lat_norm"), col("aq_lon_norm")))
    .select(
        coalesce(col("source"), lit("purpleair")).alias("aq_source"),
        coalesce(col("sensor_id"), col("name")).alias("aq_location_id"),
        col("name").alias("aq_location_name"),
        col("aq_event_ts"),
        col("kafka_timestamp"),
        col("aq_lat_norm").alias("aq_lat"),
        col("aq_lon_norm").alias("aq_lon"),
        col("h3_index"),
        col("aq_pm25_norm").alias("aq_pm25_ugm3"),
        col("temperature").cast(DoubleType()).alias("aq_temperature"),
        col("humidity").cast(DoubleType()).alias("aq_humidity"),
    )
    .filter(col("aq_event_ts").isNotNull())
)

air_quality_df = (
    openaq_df
    .unionByName(purpleair_df, allowMissingColumns=True)
    .filter(col("aq_location_id").isNotNull())
    .filter(col("aq_event_ts").isNotNull())
    .filter(col("aq_lat").isNotNull() & col("aq_lon").isNotNull())
    .filter(
        (col("aq_lat") >= NYC_LAT_MIN)
        & (col("aq_lat") <= NYC_LAT_MAX)
        & (col("aq_lon") >= NYC_LON_MIN)
        & (col("aq_lon") <= NYC_LON_MAX)
    )
    .filter(col("aq_pm25_ugm3").isNotNull() & col("aq_pm25_ugm3").between(0.0, AQ_PM25_MAX_UGM3))
    .filter(col("h3_index").isNotNull())
    .withWatermark("aq_event_ts", WATERMARK_DELAY)
)

weather_df = (
    weather_raw_df
    .withColumn("weather_event_ts", to_timestamp(col("startTime")).cast(TimestampType()))
    .withColumn("weather_join_key", lit("nyc"))
    .withColumn("weather_period_name", coalesce(col("name"), lit("Unknown")))
    .withColumn("weather_temperature", expr("TRY_CAST(temperature AS INT)"))
    .withColumn("weather_temperature_unit", coalesce(col("temperatureUnit"), lit("F")))
    .withColumn("weather_wind_speed_mph", coalesce(expr("TRY_CAST(regexp_extract(windSpeed, '(\\\\d+)', 1) AS INT)"), lit(-1)))
    .withColumn("weather_wind_direction", coalesce(col("windDirection"), lit("Unknown")))
    .withColumn("weather_short_forecast", coalesce(col("shortForecast"), lit("Unknown")))
    .withColumn("weather_precip_probability", coalesce(expr("TRY_CAST(probabilityOfPrecipitation.value AS INT)"), lit(0)))
    .filter(col("weather_event_ts").isNotNull())
    .withWatermark("weather_event_ts", WEATHER_WATERMARK_DELAY)
    .select(
        col("weather_event_ts"),
        col("weather_join_key"),
        col("weather_period_name"),
        col("weather_temperature"),
        col("weather_temperature_unit"),
        col("weather_wind_speed_mph"),
        col("weather_wind_direction"),
        col("weather_short_forecast"),
        col("weather_precip_probability"),
    )
)

[stream_config] stream=traffic topic=nyc_traffic_raw bootstrap=redpanda:29092
[stream_config] stream=openaq topic=nyc_openaq_raw bootstrap=redpanda:29092
[stream_config] stream=purpleair topic=nyc_purpleair_raw bootstrap=redpanda:29092
[stream_config] stream=weather topic=nyc_weather_raw bootstrap=redpanda:29092
[stream_config] watermark_delay=15 minutes
[stream_config] weather_watermark_delay=15 minutes
[join_config] traffic_aq_window_minutes=15
[join_config] weather_window_minutes=15
[stream_config] trigger_interval=30 seconds
[stream_config] business_checkpoint=s3a://business-data/checkpoints/local.db.enriched_traffic_v4


In [7]:
traffic_df.printSchema()
air_quality_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- speed: double (nullable = true)
 |-- travel_time: double (nullable = true)
 |-- link_name: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- from_street: string (nullable = true)
 |-- to_street: string (nullable = true)
 |-- data_as_of: string (nullable = true)
 |-- link_points: string (nullable = true)
 |-- encoded_poly_line: string (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- first_link_point: string (nullable = true)
 |-- traffic_lat: double (nullable = true)
 |-- traffic_lon: double (nullable = true)
 |-- traffic_geom: geometry (nullable = true)
 |-- traffic_event_ts: timestamp (nullable = true)
 |-- h3_index: string (nullable = true)

root
 |-- aq_source: string (nullable = false)
 |-- aq_location_id: string (nullable = true)
 |-- aq_location_name: string (nullable = true)
 |-- aq_event_ts: timestamp (nullable = true)
 |-- kafka_timestamp: timestamp (nul

In [8]:
from pyspark.sql.functions import broadcast

print("Loading Static Geospatial Datasets...")

# Load Congestion Zone (GeoJSON)
congestion_df = (
    spark.read.option("multiline", "true").json("s3a://raw-data/static/congestion_zones/")
    .selectExpr("explode(features) as feature")
    .selectExpr("ST_GeomFromGeoJSON(to_json(feature.geometry)) as zone_geom", "True as is_in_congestion_zone")
)

# Load Truck Routes (Shapefile)
truck_routes_df = (
    spark.read.format("shapefile")
    .load("s3a://raw-data/static/truck_routes/")
    .selectExpr("geometry as route_geom", "True as is_near_truck_route")
)

# Load LL84 Building Energy Data (CSV)
# We convert the lat/lon into a Sedona Geometry point for distance checking
# 3. Load LL84 Building Energy Data (CSV)
ll84_df = (
    spark.read.option("header", "true").csv("s3a://raw-data/static/building_energy/*.csv")
    .filter(col("Latitude").isNotNull() & col("Longitude").isNotNull())
    .selectExpr(
        "ST_Point(cast(Longitude as double), cast(Latitude as double)) as bldg_geom", 
        "TRY_CAST(`Total GHG Emissions (Metric Tons CO2e)` as double) as bldg_ghg_emissions"
    )
    .filter(col("bldg_ghg_emissions") > 10000.0) 
    .withColumn("is_near_high_emission_bldg", lit(True))
)

print("Building reusable H3 static lookup table...")

# Idempotent seed generation: each run recomputes a distinct H3 key set from raw traffic.
# This avoids duplicate lookup entries across notebook re-runs.
traffic_h3_seed_df = (
    spark.read.parquet("s3a://raw-data/traffic/")
    .selectExpr("trim(element_at(split(link_points, ' '), 1)) as first_link_point")
    .selectExpr(
        "cast(element_at(split(first_link_point, ','), 1) as double) as traffic_lat",
        "cast(element_at(split(first_link_point, ','), 2) as double) as traffic_lon",
    )
    .filter(col("traffic_lat").isNotNull() & col("traffic_lon").isNotNull())
    .filter(
        (col("traffic_lat") >= NYC_LAT_MIN)
        & (col("traffic_lat") <= NYC_LAT_MAX)
        & (col("traffic_lon") >= NYC_LON_MIN)
        & (col("traffic_lon") <= NYC_LON_MAX)
    )
    .withColumn("h3_index", latlon_to_h3(col("traffic_lat"), col("traffic_lon")))
    .filter(col("h3_index").isNotNull())
    .dropDuplicates(["h3_index"])
    .withColumn("h3_point_geom", expr("ST_Point(traffic_lon, traffic_lat)"))
)

static_h3_lookup_df = (
    traffic_h3_seed_df.alias("h")
    .join(broadcast(congestion_df.alias("c")), expr("ST_Contains(c.zone_geom, h.h3_point_geom)"), "leftOuter")
    .join(broadcast(truck_routes_df.alias("r")), expr("ST_Distance(r.route_geom, h.h3_point_geom) < 0.001"), "leftOuter")
    .join(broadcast(ll84_df.alias("b")), expr("ST_Distance(b.bldg_geom, h.h3_point_geom) < 0.001"), "leftOuter")
    .groupBy("h.h3_index")
    .agg(
        expr("max(case when c.zone_geom is not null then true else false end)").alias("is_in_congestion_zone"),
        expr("max(case when r.route_geom is not null then true else false end)").alias("is_near_truck_route"),
        expr("max(case when b.bldg_geom is not null then true else false end)").alias("is_near_high_emission_bldg"),
    )
    .withColumnRenamed("h3_index", "h3_lookup_index")
    .cache()
)

_ = static_h3_lookup_df.count()
print(f"[lookup] h3_keys={static_h3_lookup_df.count()}")
print(
    "[lookup] flagged_h3_counts "
    f"congestion={static_h3_lookup_df.filter(col('is_in_congestion_zone')).count()} "
    f"truck_route={static_h3_lookup_df.filter(col('is_near_truck_route')).count()} "
    f"high_emission_bldg={static_h3_lookup_df.filter(col('is_near_high_emission_bldg')).count()}"
)
static_h3_lookup_df.orderBy(col("h3_lookup_index")).show(25, truncate=False)

Loading Static Geospatial Datasets...
Building reusable H3 static lookup table...
[lookup] h3_keys=50
[lookup] flagged_h3_counts congestion=1 truck_route=43 high_emission_bldg=1
+---------------+---------------------+-------------------+--------------------------+
|h3_lookup_index|is_in_congestion_zone|is_near_truck_route|is_near_high_emission_bldg|
+---------------+---------------------+-------------------+--------------------------+
|872a1000affffff|false                |true               |false                     |
|872a1000bffffff|false                |true               |false                     |
|872a10013ffffff|false                |true               |false                     |
|872a10018ffffff|false                |true               |false                     |
|872a10019ffffff|false                |true               |false                     |
|872a1001affffff|false                |true               |false                     |
|872a1001cffffff|false                |

In [9]:
traffic_enriched_static_df = (
    traffic_df.alias("t")
    .join(
        broadcast(static_h3_lookup_df).alias("s"),
        col("t.h3_index") == col("s.h3_lookup_index"),
        "leftOuter",
    )
    .select(
        col("t.*"),
        coalesce(col("s.is_in_congestion_zone"), lit(False)).alias("is_in_congestion_zone"),
        coalesce(col("s.is_near_truck_route"), lit(False)).alias("is_near_truck_route"),
        coalesce(col("s.is_near_high_emission_bldg"), lit(False)).alias("is_near_high_emission_bldg"),
        coalesce(col("s.is_in_congestion_zone"), lit(False)).alias("is_congestion_zone"),
        coalesce(col("s.is_near_truck_route"), lit(False)).alias("is_truck_route"),
    )
)

# --- 2. Traffic + Air Quality Join (event-time window) ---
traffic_air_df = (
    traffic_enriched_static_df.alias("t")
    .join(
        air_quality_df.alias("aq"),
        (
            (col("t.h3_index") == col("aq.h3_index")) &
            (col("aq.aq_event_ts") >= col("t.traffic_event_ts") - expr(f"INTERVAL {JOIN_WINDOW_MINUTES} MINUTES")) &
            (col("aq.aq_event_ts") <= col("t.traffic_event_ts") + expr(f"INTERVAL {JOIN_WINDOW_MINUTES} MINUTES"))
        ),
        "leftOuter"
    )
    # Preserve unmatched traffic rows and add match diagnostics flags.
    .select(
        col("t.id").alias("id"), col("t.status").alias("status"), col("t.speed").alias("speed"),
        col("t.travel_time").alias("travel_time"), col("t.link_name").alias("link_name"),
        col("t.borough").alias("borough"), col("t.from_street").alias("from_street"),
        col("t.to_street").alias("to_street"), col("t.traffic_event_ts"),
        col("t.traffic_lat"), col("t.traffic_lon"), col("t.h3_index"),
        col("t.kafka_timestamp"),
        col("t.is_in_congestion_zone"),
        col("t.is_near_truck_route"),
        col("t.is_near_high_emission_bldg"),
        col("t.is_congestion_zone"),
        col("t.is_truck_route"),
        col("aq.aq_source"), col("aq.aq_location_id"), col("aq.aq_location_name"),
        col("aq.aq_pm25_ugm3"), col("aq.aq_temperature"), col("aq.aq_humidity"),
        col("aq.aq_event_ts").cast("string").alias("aq_event_ts_str"),
        col("aq.aq_pm25_ugm3").isNotNull().alias("has_aq_match"),
    )
    .withColumn("weather_join_key", lit("nyc"))
)

# --- 3. Traffic/AQ + Weather Join (event-time window) ---
# Keep this as a pure stream-stream stateful join for correctness/stability.
enriched_traffic_df = (
    traffic_air_df.alias("ta")
    .join(
        weather_df.alias("w"),
        (
            (col("ta.weather_join_key") == col("w.weather_join_key")) &
            (col("w.weather_event_ts") >= col("ta.traffic_event_ts") - expr(f"INTERVAL {WEATHER_JOIN_WINDOW_MINUTES} MINUTES")) &
            (col("w.weather_event_ts") <= col("ta.traffic_event_ts") + expr(f"INTERVAL {WEATHER_JOIN_WINDOW_MINUTES} MINUTES"))
        ),
        "leftOuter"
    )
    .select(
        col("ta.id").alias("traffic_id"), col("ta.status").alias("traffic_status"),
        col("ta.speed").alias("traffic_speed"), col("ta.travel_time").alias("traffic_travel_time"),
        col("ta.link_name").alias("traffic_link_name"), col("ta.borough").alias("traffic_borough"),
        col("ta.from_street").alias("traffic_from_street"), col("ta.to_street").alias("traffic_to_street"),
        col("ta.traffic_event_ts"), col("ta.traffic_lat"), col("ta.traffic_lon"), col("ta.h3_index"),
        col("ta.is_in_congestion_zone"),
        col("ta.is_near_truck_route"),
        col("ta.is_near_high_emission_bldg"),
        col("ta.is_congestion_zone"),
        col("ta.is_truck_route"),
        col("ta.aq_source"), col("ta.aq_location_id"), col("ta.aq_location_name"),
        col("ta.aq_pm25_ugm3"), col("ta.aq_temperature"), col("ta.aq_humidity"),
        to_timestamp(col("ta.aq_event_ts_str")).alias("aq_event_ts"),
        col("ta.has_aq_match"),
        col("w.weather_period_name"),
        col("w.weather_temperature"),
        col("w.weather_temperature_unit"),
        col("w.weather_wind_speed_mph"),
        col("w.weather_wind_direction"),
        col("w.weather_short_forecast"),
        col("w.weather_precip_probability"),
        col("w.weather_event_ts"),
        col("w.weather_event_ts").isNotNull().alias("has_weather_match"),
    )
)

In [10]:
raw_traffic_query = traffic_raw_df.writeStream \
    .format("parquet") \
    .option("path", "s3a://raw-data/traffic/") \
    .option("checkpointLocation", "s3a://raw-data/checkpoints/traffic") \
    .start()

raw_weather_query = weather_raw_df.writeStream \
    .format("parquet") \
    .option("path", "s3a://raw-data/weather/") \
    .option("checkpointLocation", "s3a://raw-data/checkpoints/weather") \
    .start()

raw_openaq_query = openaq_raw_df.writeStream \
    .format("parquet") \
    .option("path", "s3a://raw-data/openaq/") \
    .option("checkpointLocation", "s3a://raw-data/checkpoints/openaq") \
    .start()

raw_purpleair_query = purpleair_raw_df.writeStream \
    .format("parquet") \
    .option("path", "s3a://raw-data/purpleair/") \
    .option("checkpointLocation", "s3a://raw-data/checkpoints/purpleair") \
    .start()

malformed_json_query = malformed_json_df.writeStream \
    .format("parquet") \
    .option("path", "s3a://raw-data/quarantine/malformed_json/") \
    .option("checkpointLocation", "s3a://raw-data/checkpoints/malformed_json") \
    .start()

In [11]:
refined_traffic_query = traffic_df.writeStream \
    .format("parquet") \
    .option("path", "s3a://refined-data/traffic/") \
    .option("checkpointLocation", "s3a://refined-data/checkpoints/traffic") \
    .start()

refined_aq_query = air_quality_df.writeStream \
    .format("parquet") \
    .option("path", "s3a://refined-data/air_quality/") \
    .option("checkpointLocation", "s3a://refined-data/checkpoints/air_quality") \
    .start()

In [12]:
# Create the Iceberg namespace before writing tables to it
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.db")

DataFrame[]

In [ ]:
business_enriched_query = enriched_traffic_df.writeStream \
    .trigger(processingTime=STREAM_TRIGGER_INTERVAL) \
    .format("iceberg") \
    .outputMode("append") \
    .option("checkpointLocation", BUSINESS_CHECKPOINT_PATH) \
    .toTable("local.db.enriched_traffic")

print("All streaming queries initiated! Hydrating Raw, Refined, and Business buckets...")
print(f"[stream_config] business checkpoint={BUSINESS_CHECKPOINT_PATH}")
print(f"[stream_config] business trigger_interval={STREAM_TRIGGER_INTERVAL}")

import time

print("[stream_metrics] polling micro-batch progress every 30 seconds")
while True:
    active_queries = spark.streams.active
    if not active_queries:
        print("[stream_metrics] no active queries; exiting monitor loop")
        break

    for q in active_queries:
        if q.exception() is not None:
            raise RuntimeError(f"[stream_metrics] query failed: {q.name} :: {q.exception()}")

        progress = q.lastProgress
        if progress:
            duration_ms = progress.get("durationMs", {}).get("triggerExecution", "n/a")
            print(
                f"[stream_metrics] name={q.name} batch_id={progress.get('batchId')} "
                f"input_rows={progress.get('numInputRows')} "
                f"processed_rps={progress.get('processedRowsPerSecond')} "
                f"trigger_execution_ms={duration_ms}"
            )

    time.sleep(30)

All streaming queries initiated! Hydrating Raw, Refined, and Business buckets...
[stream_config] business checkpoint=s3a://business-data/checkpoints/local.db.enriched_traffic_v4
[stream_config] business trigger_interval=30 seconds
[stream_metrics] polling micro-batch progress every 30 seconds
[stream_metrics] name=None batch_id=253 input_rows=0 processed_rps=0.0 trigger_execution_ms=1
[stream_metrics] name=None batch_id=852 input_rows=0 processed_rps=0.0 trigger_execution_ms=2
[stream_metrics] name=None batch_id=66 input_rows=0 processed_rps=0.0 trigger_execution_ms=2
[stream_metrics] name=None batch_id=121 input_rows=0 processed_rps=0.0 trigger_execution_ms=1
[stream_metrics] name=None batch_id=198 input_rows=0 processed_rps=0.0 trigger_execution_ms=1
[stream_metrics] name=None batch_id=52 input_rows=0 processed_rps=0.0 trigger_execution_ms=2
[stream_metrics] name=None batch_id=66 input_rows=0 processed_rps=0.0 trigger_execution_ms=1
[stream_metrics] name=None batch_id=253 input_rows=